# log-back — ex1: implement log_back from the elementwise chain rule

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `log-back`. Running the final beacon cell reports progress against the `Backprop: log_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: log_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-back"
DD_SUBTOPIC = "Backprop: log_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## log_back — quick refresher

For the elementwise forward `out = log(x)`, the local derivative is `d/dx log(x) = 1/x`. Chain rule on an elementwise op collapses to a per-position product:

```
dL/dx[i] = dL/dout[i] * 1/x[i]
        => grad_x   = grad_out / x
```

Things to notice:
- Even though `out` is in the signature, you don't NEED it — `1/x` is the cleanest form. (You could equivalently use `out` via `exp(-out)`, but it's slower and numerically worse.)
- Shape of `grad_x` always equals shape of `x` (no broadcasting in a single-arg op).
- `log(0)` and `log(negative)` are domain errors at FORWARD time; `log_back` itself is only safe where the forward was — division by zero happens if `x` has a 0 anywhere.

### Exercise 1 — implement log_back from the elementwise chain rule

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to derive log_back: grad_x = grad_out / x, no Jacobian materialized.
> Keywords: log-back, elementwise, chain-rule, reciprocal
> ```

**KCs targeted:** `log-back`, `chain-rule-elementwise`

Implement `log_back(grad_out, out, x)` — the backward fn for the forward op `out = log(x)`.

**The math.** `d/dx log(x) = 1/x`. Elementwise, the Jacobian is diagonal, so the chain rule reduces to per-position product:

```
dL/dx[i] = dL/dout[i] * (1 / x[i])
        => grad_x   = grad_out / x
```

Signature (same uniform `(grad_out, out, *fwd_args)` shape every back fn uses):

```python
def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    ...
```

**Why `out` is in the signature even though you won't use it.** The uniform back-fn signature lets the reverse-pass dispatcher call any back fn the same way: `back_fn(grad_out, out, *recipe.args, **recipe.kwargs)`. Some back fns (sigmoid_back) need `out`; log_back doesn't — but the signature is fixed so dispatch is generic.

Inputs are plain `torch.Tensor`; no autograd. Return a tensor with the same shape and float dtype as `x`.

In [ ]:
def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx log(x) = 1/x; chain rule => grad_x = grad_out * (1/x).
    # `out` is in the signature only because the uniform back-fn
    # signature requires it — we don't read it here.
    return grad_out / x


<details><summary>Solution</summary>

```python
def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx log(x) = 1/x; chain rule => grad_x = grad_out * (1/x).
    # `out` is in the signature only because the uniform back-fn
    # signature requires it — we don't read it here.
    return grad_out / x
```

**Why not `grad_out * (1/x)`.** Same answer, but `grad_out / x` is one fused op in torch's kernel scheduler — fewer intermediates, lower peak memory.

**Why `out` is unused but still passed.** Every back fn in this library has the SAME signature: `(grad_out, out, *original_fwd_args, **original_fwd_kwargs)`. The dispatcher in `backprop` doesn't know which fns need `out` (sigmoid_back uses it heavily) and which don't (log_back, relu_back). Keeping the signature uniform means dispatch is one line.

**Domain.** `log` is only defined for `x > 0`. If the forward succeeded, `x` is positive everywhere, so `1/x` is finite and safe. Otherwise the forward would already have produced NaNs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()